# CertCF Single-Dataset Probe

Run CertCF on one non-Adult dataset, then inspect validity, distance, solver metadata, failures, and the feature-level changes for individual counterfactuals.

In [ ]:
from pathlib import Path
import subprocess, sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from counterfactuals.datasets.loaders import (
    AdultDataset, CompasDataset, GermanCreditDataset, GiveMeSomeCreditDataset,
    HELOCDataset, LendingClubDataset, WisconsinBreastCancerDataset,
)

CONFIG = ROOT / "configs/benchmarks/certcf_single_dataset_probe.yaml"

DATASET_LOADERS = {
    "adult": AdultDataset,
    "compas": CompasDataset,
    "german_credit": GermanCreditDataset,
    "give_me_some_credit": GiveMeSomeCreditDataset,
    "heloc": HELOCDataset,
    "lending_club": LendingClubDataset,
    "wisconsin_breast_cancer": WisconsinBreastCancerDataset,
}

Choose one dataset. The default is `give_me_some_credit`, which has 10 encoded dimensions. Other supported choices here are `compas`, `heloc`, `lending_club`, `german_credit`, and `wisconsin_breast_cancer`.

In [ ]:
DATASET = "adult"
N_QUERIES = 1000
RUN_BENCHMARK = True
METHOD_TO_INSPECT = "certcf_probe"

result_path = ROOT / f"results/certcf_probe_{DATASET}.parquet"
if RUN_BENCHMARK:
    cmd = [
        sys.executable, str(ROOT / "scripts/benchmark.py"),
        "--config", str(CONFIG),
        # "--datasets", DATASET,
        "--n_queries", str(N_QUERIES),
        "--output", str(result_path),
    ]
    subprocess.run(cmd, cwd=ROOT, check=True)

df = pd.read_parquet(result_path)
df.shape

In [ ]:
def _cf_columns(frame):
    return sorted([c for c in frame.columns if c.startswith("x_cf_")], key=lambda c: int(c.rsplit("_", 1)[1]))


def _is_training_point_matrix(x_cf, x_train, atol=1e-6, chunk_size=4096):
    hits = np.zeros(len(x_cf), dtype=bool)
    finite = np.isfinite(x_cf).all(axis=1)
    for start in range(0, len(x_train), chunk_size):
        train_chunk = x_train[start:start + chunk_size]
        hits[finite] |= np.isclose(x_cf[finite, None, :], train_chunk[None, :, :], atol=atol, rtol=0.0).all(axis=2).any(axis=1)
    return hits


loader = DATASET_LOADERS[DATASET](data_dir=str(ROOT / "data"), seed=42)
loader.load()
x_train, _ = loader.get_train()
cf_cols = _cf_columns(df)
df = df.copy()
df["cf_is_training_point"] = np.nan
if cf_cols:
    successful = df["success"].astype(bool).to_numpy()
    x_cf = df.loc[successful, cf_cols].to_numpy(dtype=np.float32)
    df.loc[successful, "cf_is_training_point"] = _is_training_point_matrix(x_cf, x_train).astype(float)

summary = df.groupby(["dataset", "run_name"]).agg(
    n=("success", "size"),
    validity=("success", "mean"),
    returned_cfs=("cf_is_training_point", "count"),
    train_point_probability=("cf_is_training_point", "mean"),
    train_point_hits=("cf_is_training_point", "sum"),
    l1_mean=("l1_distance", "mean"),
    l1_median=("l1_distance", "median"),
    query_s=("runtime_s", "mean"),
)
for col in ["meta__nearest_anchor_returned", "meta__n_qp_solved", "meta__target_reached"]:
    if col in df.columns:
        summary[col.replace("meta__", "")] = df.groupby(["dataset", "run_name"])[col].mean()
display(summary.round(4))